In [ ]:
#PREPARACIÓN DE ENTORNO (DEPENDENCIAS, IMPORTACIÓN DE DATASETS Y LOGINS)

In [ ]:
#1. Instalar dependencias
!pip install -q torch torchvision torchaudio
!pip install -q transformers datasets scikit-learn pandas
!pip install -q peft accelerate evaluate
!pip install -q kagglehub

In [ ]:
# 2.1. Rutas de los datasets en Kaggle
import kagglehub

# Dataset ISOT
path_isot = kagglehub.dataset_download("csmalarkodi/isot-fake-news-dataset")

# Dataset FakeNewsNet
path_fnn = kagglehub.dataset_download("mdepak/fakenewsnet")

# Dataset Welfake
path_welfake = kagglehub.dataset_download("studymart/welfake-dataset-for-fake-news")

print("ISOT path:", path_isot)
print("FakeNewsNet path:", path_fnn)
print("Welfake path:", path_welfake)

Using Colab cache for faster access to the 'isot-fake-news-dataset' dataset.
Using Colab cache for faster access to the 'fakenewsnet' dataset.
Using Colab cache for faster access to the 'welfake-dataset-for-fake-news' dataset.
ISOT path: /kaggle/input/isot-fake-news-dataset
FakeNewsNet path: /kaggle/input/fakenewsnet
Welfake path: /kaggle/input/welfake-dataset-for-fake-news


In [ ]:
# 2.2. Descargar datasets de ISOT
import os

fake_csv_path = os.path.join(path_isot, "Fake.csv")
true_csv_path = os.path.join(path_isot, "True.csv")

print(fake_csv_path)
print(true_csv_path)

/kaggle/input/isot-fake-news-dataset/Fake.csv
/kaggle/input/isot-fake-news-dataset/True.csv


In [ ]:
# 2.3. Descargar datasets de FakeNewsNet
buzz_fake_path = os.path.join(path_fnn, "BuzzFeed_fake_news_content.csv")
buzz_real_path = os.path.join(path_fnn, "BuzzFeed_real_news_content.csv")
politi_fake_path = os.path.join(path_fnn, "PolitiFact_fake_news_content.csv")
politi_real_path = os.path.join(path_fnn, "PolitiFact_real_news_content.csv")

print(buzz_fake_path)
print(buzz_real_path)
print(politi_fake_path)
print(politi_real_path)

/kaggle/input/fakenewsnet/BuzzFeed_fake_news_content.csv
/kaggle/input/fakenewsnet/BuzzFeed_real_news_content.csv
/kaggle/input/fakenewsnet/PolitiFact_fake_news_content.csv
/kaggle/input/fakenewsnet/PolitiFact_real_news_content.csv


In [ ]:
# 2.4. Descargar dataset de Welfake
welfake_csv_path = os.path.join(path_welfake, "WELFake_Dataset.csv")

print(welfake_csv_path)

/kaggle/input/welfake-dataset-for-fake-news/WELFake_Dataset.csv


In [ ]:
#3. Login en Hugging Face
from google.colab import userdata
from huggingface_hub import login

# Obtener secreto de Colab
token = userdata.get("HUGGINGFACE_TOKEN")

# Login en Hugging Face
login(token=token)

print("Token cargado correctamente:", token is not None)

Token cargado correctamente: True


In [ ]:
#IMPLEMENTACIÓN DEL MODELO

In [ ]:
#4. Imports
import pandas as pd
import torch
import numpy as np
from sklearn.model_selection import train_test_split

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
import evaluate

In [ ]:
# 5.1. Función de carga de datasets genérica
def load_dataset(path, label):
    df = pd.read_csv(path)
    df = df[['title', 'text']].dropna()
    df['label'] = label
    return df

In [ ]:
# 5.1.1 Función de carga de dataset específica para el dataset WELFake
# Dataset original:
#   0 = fake
#   1 = real
#
# Dataset entrenamiento:
#   0 = real
#   1 = fake

def load_dataset_welfake(path):
    df = pd.read_csv(path)

    # Nos quedamos con las columnas necesarias
    df = df[['title', 'text', 'label']].dropna()

    # Invertir labels:
    # original 1 -> nuevo 0
    # original 0 -> nuevo 1
    df['label'] = df['label'].apply(lambda x: 0 if x == 1 else 1)

    return df

In [ ]:
# 5.2. Cargar datasets ISOT y recortar a 3000 filas
df_fake = load_dataset(fake_csv_path, 1).head(3000)
df_real = load_dataset(true_csv_path, 0).head(3000)

# 5.3. Cargar datasets FakeNewsNet completos
df_buzz_fake = load_dataset(buzz_fake_path, 1)
df_buzz_real = load_dataset(buzz_real_path, 0)
df_politi_fake = load_dataset(politi_fake_path, 1)
df_politi_real = load_dataset(politi_real_path, 0)

# 5.4. Cargar dataset Welfake y recortar a 3000 filas
df_welfake = load_dataset_welfake(welfake_csv_path).head(3000)

In [ ]:
#Mostrar cabecera de cada datasets que conformará mi dataset para entrenamiento del modelo

In [ ]:
df_fake.head()

,title,text,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,1
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,1
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",1
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",1
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,1


In [ ]:
df_real.head()

,title,text,label
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,0
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,0
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,0
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,0
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,0


In [ ]:
df_buzz_fake.head()

,title,text,label
0,Proof The Mainstream Media Is Manipulating The...,I woke up this morning to find a variation of ...,1
1,Charity: Clinton Foundation Distributed “Water...,Former President Bill Clinton and his Clinton ...,1
2,A Hillary Clinton Administration May be Entire...,After collapsing just before trying to step in...,1
3,Trump’s Latest Campaign Promise May Be His Mos...,"Donald Trump is, well, deplorable. He’s sugges...",1
4,Website is Down For Maintenance,Website is Down For Maintenance,1


In [ ]:
df_buzz_real.head()

,title,text,label
0,Another Terrorist Attack in NYC…Why Are we STI...,"On Saturday, September 17 at 8:30 pm EST, an e...",0
1,"Donald Trump: Drugs a 'Very, Very Big Factor' ...",Less than a day after protests over the police...,0
2,"Obama To UN: ‘Giving Up Liberty, Enhances Secu...","Obama To UN: ‘Giving Up Liberty, Enhances Secu...",0
3,Trump vs. Clinton: A Fundamental Clash over Ho...,Getty Images Wealth Of Nations Trump vs. Clint...,0
4,"President Obama Vetoes 9/11 Victims Bill, Sett...",President Obama today vetoed a bill that would...,0


In [ ]:
df_politi_fake.head()

,title,text,label
0,Trump Just Insulted Millions Who Lost Everythi...,16.8k SHARES SHARE THIS STORY\n\nHillary Clint...,1
1,Famous dog killed in spot she waited a year fo...,Famous dog killed in spot she waited a year fo...,1
2,House oversight panel votes Clinton IT chief i...,Story highlights The House Oversight panel vot...,1
3,America Just Tragically Lost A Country Music I...,We are absolutely heartbroken to hear about th...,1
4,Monuments to the Battle for the New South,"Nine years ago, a driver lost control of his p...",1


In [ ]:
df_politi_real.head()

,title,text,label
0,Trump Just Insulted Millions Who Lost Everythi...,16.8k SHARES SHARE THIS STORY\n\nHillary Clint...,0
1,Famous dog killed in spot she waited a year fo...,Famous dog killed in spot she waited a year fo...,0
2,House oversight panel votes Clinton IT chief i...,Story highlights The House Oversight panel vot...,0
3,America Just Tragically Lost A Country Music I...,We are absolutely heartbroken to hear about th...,0
4,Monuments to the Battle for the New South,"Nine years ago, a driver lost control of his p...",0


In [ ]:
df_welfake.head()

,title,text,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,0
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",0
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,1
4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",0
5,About Time! Christian Group Sues Amazon and SP...,All we can say on this one is it s about time ...,0


In [ ]:
# 5.4. Unir todos los datasets en uno solo
df_total = pd.concat([
    df_fake,
    df_real,
    df_buzz_fake,
    df_buzz_real,
    df_politi_fake,
    df_politi_real,
    df_welfake
], ignore_index=True)

# 5.5. Crear columna "content" concatenando título + texto
df_total["content"] = df_total["title"] + " " + df_total["text"]

# 5.6. Comprobar distribución de etiquetas
print(df_total["label"].value_counts())

# 5.7. Mostrar las primeras filas de mi dataset
df_total.head()

label
0    4797
1    4625
Name: count, dtype: int64


,title,text,label,content
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,1,Donald Trump Sends Out Embarrassing New Year’...
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,1,Drunk Bragging Trump Staffer Started Russian ...
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",1,Sheriff David Clarke Becomes An Internet Joke...
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",1,Trump Is So Obsessed He Even Has Obama’s Name...
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,1,Pope Francis Just Called Out Donald Trump Dur...


In [ ]:
# 6. Split train / validation / test (Dividir el dataset en entrenamiento, validación y prueba)
def random_split(df, train_frac, validation_frac):
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)

    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df

In [ ]:
# 6.1. Obtener un dataset para entrenamiento, otro para validación y otro para prueba
train_df, validation_df, test_df = random_split(df_total, 0.7, 0.1)

# Guardar los CSV
train_df.to_csv("train.csv", index=False)
validation_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)

# Revisar el tamaño de cada dataset
print("Train:", len(train_df), "Validation:", len(validation_df), "Test:", len(test_df))

Train: 6595 Validation: 942 Test: 1885


In [ ]:
#7. Creación de DataLoaders
from transformers import AutoTokenizer

model_name = "google/gemma-3-1b-pt"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Gemma no tiene pad token por defecto → lo igualamos al eos
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
#8. Crear dataset personalizado (CustomDataset)
import torch
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=256):
        self.data = pd.read_csv(csv_file)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        text = self.data.iloc[index]["content"]
        label = self.data.iloc[index]["label"]

        # Sin padding fijo → se hace dinámicamente en el DataCollator
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }


In [ ]:
#9. Crear datasets usando (CustomDataset)
train_dataset = CustomDataset(
    csv_file="train.csv",
    tokenizer=tokenizer,
    max_length=512
)

val_dataset = CustomDataset(
    csv_file="validation.csv",
    tokenizer=tokenizer,
    max_length=512
)

test_dataset = CustomDataset(
    csv_file="test.csv",
    tokenizer=tokenizer,
    max_length=512
)

In [ ]:
#10. DataLoaders con padding dinámico
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

batch_size = 4

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=data_collator
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=data_collator
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    collate_fn=data_collator
)


In [ ]:
#11. Verificación de dimensiones
for batch in train_loader:
    break

print("Input shape:", batch["input_ids"].shape)
print("Attention mask shape:", batch["attention_mask"].shape)
print("Labels shape:", batch["labels"].shape)

Input shape: torch.Size([4, 512])
Attention mask shape: torch.Size([4, 512])
Labels shape: torch.Size([4])


In [ ]:
#12. Verificación de formato
batch["input_ids"].dtype
batch["labels"].dtype

torch.int64

In [ ]:
#13. (DEMO) Inicializamos el modelo generativo base para ver capacidad zero-shot
# NOTA: Este modelo (CausalLM) es SOLO para la demostración generativa de las celdas 14 y 15.
# El modelo que se entrenará para clasificación se carga en la celda 16 con AutoModelForSequenceClassification.
from transformers import AutoModelForCausalLM

model_name = "google/gemma-3-1b-pt"

base_model_demo = AutoModelForCausalLM.from_pretrained(model_name)
base_model_demo.eval()


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1152, out_features=256, bias=False)
          (v_proj): Linear(in_features=1152, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((1152,), e

In [ ]:
#14. Comprobación generativa usando el modelo que acabamos de cargar
input_text = "Every effort moves you"
inputs = tokenizer(input_text, return_tensors="pt")

with torch.no_grad():
    outputs = base_model_demo.generate(
        **inputs,
        max_new_tokens=20
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Every effort moves you forward, so don’t stand still. We know you have busy schedules, so get us to


In [ ]:
#15. Ejemplo de tipo clasificación de fake news usando prompting
text_2 = (
    "Is the following news fake? Answer with 'yes' or 'no': "
    "'Breaking: Scientists confirm the Earth is flat after new NASA study.'"
)

inputs = tokenizer(text_2, return_tensors="pt")

with torch.no_grad():
    outputs = base_model_demo.generate(
        **inputs,
        max_new_tokens=20
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Is the following news fake? Answer with 'yes' or 'no': 'Breaking: Scientists confirm the Earth is flat after new NASA study.'

This news comes from The Daily Mail.

The Daily Mail reports,

<blockquote>A former astronomer


El modelo base muestra capacidad de razonamiento semántico mediante prompting zero-shot, aunque no produce respuestas estrictamente estructuradas. Por ello, se procede a adaptar el modelo mediante fine-tuning supervisado con cabeza de clasificación.

In [ ]:
#16. Cargar modelo con cabeza de clasificación (AutoModelForSequenceClassification)
# AutoModelForSequenceClassification = backbone de Gemma + cabeza Linear(hidden_size, num_labels)
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "REAL", 1: "FAKE"},
    label2id={"REAL": 0, "FAKE": 1},
)

# Gemma no tiene pad_token_id por defecto → hay que indicárselo al modelo también
model.config.pad_token_id = tokenizer.pad_token_id

print("Modelo cargado. Cabeza clasificadora:", model.score)
print("Arquitectura: backbone Gemma3 + Linear(", model.config.hidden_size, ", 2)")


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Gemma3TextForSequenceClassification LOAD REPORT from: google/gemma-3-1b-pt
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modelo cargado. Cabeza clasificadora: Linear(in_features=1152, out_features=2, bias=False)
Arquitectura: backbone Gemma3 + Linear( 1152 , 2)


In [ ]:
#   - Backbone Gemma congelable capa a capa
#   - model.score = nn.Linear(hidden_size, num_labels)  ← la cabeza clasificadora
#   - Internamente usa el último token para clasificar

In [ ]:
#18. Verificamos estructura del modelo instanciado
print("Tipo de modelo:", type(model))
print("Cabeza clasificadora (model.score):", model.score)
print("hidden_size:", model.config.hidden_size)
print("num_labels:", model.config.num_labels)
print("id2label:", model.config.id2label)


Tipo de modelo: <class 'transformers.models.gemma3.modeling_gemma3.Gemma3TextForSequenceClassification'>
Cabeza clasificadora (model.score): Linear(in_features=1152, out_features=2, bias=False)
hidden_size: 1152
num_labels: 2
id2label: {0: 'REAL', 1: 'FAKE'}


In [ ]:
#20. Descongelar últimas N capas y la cabeza clasificadora
# En AutoModelForSequenceClassification:
#   model.model  → backbone Gemma (todas las capas transformer)
#   model.score  → cabeza Linear(hidden_size, 2)
#   model.model.layers  → lista de capas del transformer

N = 5  # Número de últimas capas a descongelar

# 1. Congelar todo el backbone
for param in model.model.parameters():
    param.requires_grad = False

# 2. Descongelar las últimas N capas del transformer
for layer in model.model.layers[-N:]:
    for param in layer.parameters():
        param.requires_grad = True

# 3. La cabeza (model.score) siempre entrenada
for param in model.score.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Parámetros entrenables: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")


Parámetros entrenables: 134,212,864 / 999,888,256 (13.42%)


In [ ]:
#21. Verificación dimensional
batch = next(iter(train_loader))

input_ids = batch["input_ids"]
attention_mask = batch["attention_mask"]

with torch.no_grad():
    outputs = model(input_ids, attention_mask)

print("Output shape:", outputs.logits.shape)  # ← .logits para acceder al tensor
print("Ejemplo de logits:", outputs.logits[0])  # logits del primer elemento del batch

Output shape: torch.Size([4, 2])
Ejemplo de logits: tensor([-1.2969,  2.7812], dtype=torch.bfloat16)


In [ ]:
#22. Obtener probabilidades y clase predicha
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)

with torch.no_grad():
    outputs = model(input_ids, attention_mask)  # ← outputs, no logits directamente
    logits = outputs.logits                      # ← extraer .logits del objeto

probas = torch.softmax(logits, dim=-1)
predicted_label = torch.argmax(probas, dim=-1)

print("Device:", device)
print("Predicted labels:", predicted_label)
print("True labels:     ", labels)

Device: cuda
Predicted labels: tensor([1, 1, 1, 0], device='cuda:0')
True labels:      tensor([0, 1, 1, 1], device='cuda:0')


In [ ]:
#23. Función de accuracy adaptada para AutoModelForSequenceClassification
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, batch in enumerate(data_loader):
        if i >= num_batches:
            break

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.no_grad():
            # AutoModelForSequenceClassification devuelve SequenceClassifierOutput
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits  # ← accedemos al atributo .logits

        predicted_labels = torch.argmax(logits, dim=-1)
        num_examples += labels.size(0)
        correct_predictions += (predicted_labels == labels).sum().item()

    return correct_predictions / num_examples


In [ ]:
#24. Accuracy inicial (antes de entrenar)
train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=10)
val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=10)
test_accuracy = calc_accuracy_loader(test_loader, model, device, num_batches=10)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

Training accuracy: 67.50%
Validation accuracy: 55.00%
Test accuracy: 52.50%


In [ ]:
#25. Loss batch adaptado para AutoModelForSequenceClassification
def calc_loss_batch(batch, model, device):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    # Pasamos labels directamente → el modelo calcula la loss internamente
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels  # ← AutoModelForSequenceClassification acepta labels y calcula cross_entropy solo
    )

    return outputs.loss


In [ ]:
#26. Loss loader adaptado
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.

    if len(data_loader) == 0:
        return float("nan")

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, batch in enumerate(data_loader):
        if i >= num_batches:
            break

        loss = calc_loss_batch(batch, model, device)
        total_loss += loss.item()

    return total_loss / num_batches

In [ ]:
#27. Evaluación inicial de loss
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
    test_loss = calc_loss_loader(test_loader, model, device, num_batches=5)

print(f"Training loss: {train_loss:.3f}")
print(f"Validation loss: {val_loss:.3f}")
print(f"Test loss: {test_loss:.3f}")

Training loss: 1.542
Validation loss: 0.951
Test loss: 0.711


In [ ]:
#FINETUNING EL MODELO

In [ ]:
#28. Training loop con Best Model Checkpoint
def train_classifier_simple(model, train_loader, val_loader, optimizer, device,
                            num_epochs, eval_freq, eval_iter, scheduler=None,
                            save_path="best_model_state.pt"):

    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    examples_seen, global_step = 0, -1

    best_val_loss = float("inf")

    for epoch in range(num_epochs):
        model.train()

        for batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(batch, model, device)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            if scheduler is not None:
                scheduler.step()

            examples_seen += batch["input_ids"].size(0)
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter
                )
                train_losses.append(train_loss)
                val_losses.append(val_loss)

                current_lr = scheduler.get_last_lr()[0] if scheduler else optimizer.param_groups[0]['lr']
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}, LR {current_lr:.2e}")

                # ── Checkpoint: guardar solo si mejora ────────────────────
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    torch.save(model.state_dict(), save_path)
                    print(f"  ✓ Nuevo mejor modelo — val_loss: {best_val_loss:.3f}")
                # ─────────────────────────────────────────────────────────

        train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=eval_iter)
        val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=eval_iter)

        print(f"Training accuracy: {train_accuracy*100:.2f}% | "
              f"Validation accuracy: {val_accuracy*100:.2f}%")

        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)

    # ── Al terminar todas las épocas, restaurar el mejor modelo ──────────
    import os
    if os.path.exists(save_path):
        model.load_state_dict(torch.load(save_path, map_location=device))
        print(f"\n✓ Pesos restaurados al mejor modelo (val_loss: {best_val_loss:.3f})")
        os.remove(save_path)

    return train_losses, val_losses, train_accs, val_accs, examples_seen

In [ ]:
#29. Función de evaluación
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

In [ ]:
#30. Optimizer y entrenamiento con scheduler cosine
import time
import torch
from transformers import get_cosine_schedule_with_warmup

start_time = time.time()
torch.manual_seed(123)

# Contar parámetros entrenables
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parámetros entrenables: {trainable_params}/{total_params} ({trainable_params/total_params*100:.2f}%)")

num_epochs = 3

# weight_decay reducido (0.01 es el estándar para LLMs con pocas capas)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    #lr=1e-5,
    lr=2e-5,
    weight_decay=0.01
)

# Scheduler coseno con warmup: la LR sube suavemente al inicio y baja al final
total_steps = num_epochs * len(train_loader)
warmup_steps = int(0.1 * total_steps)  # 10% de warmup

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f"Total steps: {total_steps}, Warmup steps: {warmup_steps}")

train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs,
    eval_freq=50,
    eval_iter=100,
    scheduler=scheduler,
    save_path="best_model_state.pt",
)

end_time = time.time()
print(f"Training completed in {(end_time - start_time)/60:.2f} minutes.")


Parámetros entrenables: 134212864/999888256 (13.42%)
Total steps: 4947, Warmup steps: 494
Ep 1 (Step 000000): Train loss 1.158, Val loss 1.032, LR 4.05e-08
  ✓ Nuevo mejor modelo — val_loss: 1.032
Ep 1 (Step 000050): Train loss 1.079, Val loss 1.073, LR 2.06e-06
Ep 1 (Step 000100): Train loss 0.839, Val loss 0.946, LR 4.09e-06
  ✓ Nuevo mejor modelo — val_loss: 0.946
Ep 1 (Step 000150): Train loss 0.891, Val loss 0.832, LR 6.11e-06
  ✓ Nuevo mejor modelo — val_loss: 0.832
Ep 1 (Step 000200): Train loss 0.786, Val loss 0.806, LR 8.14e-06
  ✓ Nuevo mejor modelo — val_loss: 0.806
Ep 1 (Step 000250): Train loss 0.804, Val loss 0.888, LR 1.02e-05
Ep 1 (Step 000300): Train loss 0.787, Val loss 0.803, LR 1.22e-05
  ✓ Nuevo mejor modelo — val_loss: 0.803
Ep 1 (Step 000350): Train loss 0.784, Val loss 0.732, LR 1.42e-05
  ✓ Nuevo mejor modelo — val_loss: 0.732
Ep 1 (Step 000400): Train loss 0.630, Val loss 0.707, LR 1.62e-05
  ✓ Nuevo mejor modelo — val_loss: 0.707
Ep 1 (Step 000450): Train los

In [ ]:
#31. Accuracy final
train_accuracy = calc_accuracy_loader(train_loader, model, device)
val_accuracy = calc_accuracy_loader(val_loader, model, device)
test_accuracy = calc_accuracy_loader(test_loader, model, device)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

Training accuracy: 88.40%
Validation accuracy: 85.14%
Test accuracy: 84.19%


In [ ]:
# 32. Guardar y subir a Hugging Face con save_pretrained + push_to_hub
# Con AutoModelForSequenceClassification todo se guarda en el formato estándar HuggingFace
import os

save_dir = "gemma_fakenews_classifier"
os.makedirs(save_dir, exist_ok=True)

# ── 1. Guardar modelo y tokenizer en local ─────────────────────────────────
model.save_pretrained(save_dir)       # guarda config.json + model.safetensors
tokenizer.save_pretrained(save_dir)   # guarda tokenizer_config.json + tokenizer.model
print(f"Modelo guardado en '{save_dir}'")
print("Archivos:", os.listdir(save_dir))

# ── 2. Subir a Hugging Face Hub ───────────────────────────────────────────
HF_USERNAME = "MartaAguilarMorcillo"
REPO_NAME = "gemma-fakenews-classifier"

# push_to_hub sube: pesos, config y tokenizer
model.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}")
tokenizer.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}")

print(f"\nModelo subido a: https://huggingface.co/{HF_USERNAME}/{REPO_NAME}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo guardado en 'gemma_fakenews_classifier'
Archivos: ['model.safetensors', 'tokenizer.json', 'config.json', 'tokenizer_config.json']


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpmdqrtc9c/tokenizer.json: 100%|##########| 33.4MB / 33.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.



Modelo subido a: https://huggingface.co/MartaAguilarMorcillo/gemma-fakenews-classifier


In [ ]:
# 33. Helper de inferencia para la API (backend)
# Carga y uso de mi modelo

def load_classifier_from_hub(repo_id, device="cpu"):
    """Carga el clasificador desde Hugging Face Hub.
    Uso: model, tokenizer = load_classifier_from_hub('MartaAguilarMorcillo/gemma-fakenews-classifier')
    """
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(repo_id)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForSequenceClassification.from_pretrained(repo_id)
    model.to(device)
    model.eval()

    return model, tokenizer


def predict(text, model, tokenizer, device="cpu", max_length=256):
    """Predice si una noticia es REAL (0) o FAKE (1).
    Retorna: {'label': 'FAKE'|'REAL', 'confidence': float, 'probas': {'REAL': float, 'FAKE': float}}
    """
    import torch
    inputs = tokenizer(
        text,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probas = torch.softmax(outputs.logits, dim=-1)
        predicted = torch.argmax(probas, dim=-1).item()

    label = model.config.id2label[predicted]  # usa el id2label de la config
    confidence = probas[0][predicted].item()

    return {
        "label": label,
        "confidence": round(confidence, 4),
        "probas": {
            "REAL": round(probas[0][0].item(), 4),
            "FAKE": round(probas[0][1].item(), 4)
        }
    }

print("Helper de inferencia listo")


Helper de inferencia listo
